In [ ]:
from pyspark.sql.functions import col, current_timestamp, to_timestamp

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_order_items_table_name = dbutils.widgets.get("raw_olist_order_items_table")

silver_schema = dbutils.widgets.get("silver_schema")
order_items_table_name = dbutils.widgets.get("order_items_table")

In [ ]:
raw_olist_order_items_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_order_items_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{order_items_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{order_items_table_name} (
            orderId STRING,
            orderItemId INT,
            productId STRING,
            sellerId STRING,
            shippingLimitDate TIMESTAMP,
            price DOUBLE,
            freightValue DOUBLE,
            processedTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
order_items_silver_df = (
    raw_olist_order_items_df.where(
        (col("order_id").rlike("^[0-9a-fA-F]{32}$"))
        & (col("product_id").rlike("^[0-9a-fA-F]{32}$"))
        & (col("seller_id").rlike("^[0-9a-fA-F]{32}$"))
        & (col("order_item_id").cast("int").isNotNull())
    )
    .select(
        col("order_id").cast("string").alias("orderId"),
        col("order_item_id").cast("int").alias("orderItemId"),
        col("product_id").cast("string").alias("productId"),
        col("seller_id").cast("string").alias("sellerId"),
        to_timestamp(col("shipping_limit_date"), "yyyy-MM-dd HH:mm:ss").alias("shippingLimitDate"),
        col("price").cast("double").alias("price"),
        col("freight_value").cast("double").alias("freightValue"),
    )
    .withColumn("processedTimestamp", current_timestamp())
    .dropDuplicates(["orderId", "orderItemId"])
)

In [ ]:
order_items_silver_df.createOrReplaceTempView("order_items_silver_view")

spark.sql(f"""
    MERGE INTO {catalog}.{silver_schema}.{order_items_table_name} AS target
    USING order_items_silver_view AS source
    ON target.orderId = source.orderId AND target.orderItemId = source.orderItemId
    WHEN MATCHED THEN
        UPDATE SET
            target.productId = source.productId,
            target.sellerId = source.sellerId,
            target.shippingLimitDate = source.shippingLimitDate,
            target.price = source.price,
            target.freightValue = source.freightValue,
            target.processedTimestamp = source.processedTimestamp
    WHEN NOT MATCHED THEN
        INSERT *
    """)